# Data Acquisition

Note: This notebook contains both files that I downloaded directly from a website and files that I downloaded through using various packages such as: `requests`, `zipfile`, and `io`.

In [21]:
import pandas as pd
import requests
import zipfile
import io

## OSB

Downloaded directly from: https://open.canada.ca/data/en/dataset/746709f1-c729-44a1-ba84-7be5eadd3664/resource/cc792c91-6d97-49cd-ac9f-005aad997637

## BOC

In [22]:
url = "https://www.bankofcanada.ca/valet/observations/V39079"
response = requests.get(url, params={'start_date': '2010-01-01'})
data = response.json()

# Inspect the structure first
print(data.keys())  # See what top-level keys exist
print(data['observations'][0])  # See what ONE observation actually looks like

dict_keys(['terms', 'seriesDetail', 'observations'])
{'d': '2010-01-01', 'V39079': {'v': '0.25'}}


In [23]:
url = "https://www.bankofcanada.ca/valet/observations/V39079"
response = requests.get(url, params={'start_date': '2010-01-01'})
data = response.json()

# Each observation looks like: {'d': '2010-01-05', 'V39079': {'v': '0.25'}}
observations = data['observations']

# Extract date and rate value explicitly
rows = []
for obs in observations:
    rows.append({
        'date': obs['d'],                      # 'd' not 'date'
        'overnight_rate': obs['V39079']['v']   # nested under series ID
    })

boc_data = pd.DataFrame(rows)
boc_data['date'] = pd.to_datetime(boc_data['date'])
boc_data['overnight_rate'] = pd.to_numeric(boc_data['overnight_rate'], errors='coerce')
boc_data = boc_data.sort_values('date').reset_index(drop=True)

print(boc_data.head(10))
print(boc_data.shape)

        date  overnight_rate
0 2010-01-01            0.25
1 2010-01-04            0.25
2 2010-01-05            0.25
3 2010-01-06            0.25
4 2010-01-07            0.25
5 2010-01-08            0.25
6 2010-01-11            0.25
7 2010-01-12            0.25
8 2010-01-13            0.25
9 2010-01-14            0.25
(4250, 2)


In [26]:
boc_data.to_csv('../data/raw/03_boc_daily.csv', index=False)
print("✓ Saved to ../data/raw/03_boc_daily.csv")

✓ Saved to ../data/raw/03_boc_daily.csv


## StatsCan

### CPI

Downloaded directly from: https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1810000401&pickMembers%5B0%5D=1.2&cubeTimeFrame.startMonth=01&cubeTimeFrame.startYear=2010&cubeTimeFrame.endMonth=05&cubeTimeFrame.endYear=2026&referencePeriods=20100101%2C20260501

### Unemployment

In [ ]:
# Step 1: get the zip URL
r = requests.get("https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/14100287/en")
zip_url = r.json()["object"]
print(zip_url)

# Step 2: stream the zip and filter on the fly
r2 = requests.get(zip_url, stream=True)
z = zipfile.ZipFile(io.BytesIO(r2.content))

# find the data file (not the metadata file)
csv_name = [f for f in z.namelist() if not "MetaData" in f][0]
print(csv_name)

# Step 3: read in chunks, filter to 2010+, write filtered output
chunks = []
with z.open(csv_name) as f:
    for chunk in pd.read_csv(f, chunksize=50000):
        filtered = chunk[chunk["REF_DATE"] >= "2010-01"]
        chunks.append(filtered)

https://www150.statcan.gc.ca/n1/tbl/csv/14100287-eng.zip
14100287.csv


C:\Users\karle\AppData\Local\Temp\ipykernel_33184\1032639278.py:22: DtypeWarning: Columns (0: STATUS) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f, chunksize=50000):
C:\Users\karle\AppData\Local\Temp\ipykernel_33184\1032639278.py:22: DtypeWarning: Columns (0: STATUS) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f, chunksize=50000):
C:\Users\karle\AppData\Local\Temp\ipykernel_33184\1032639278.py:22: DtypeWarning: Columns (0: STATUS) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f, chunksize=50000):
C:\Users\karle\AppData\Local\Temp\ipykernel_33184\1032639278.py:22: DtypeWarning: Columns (0: STATUS) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f, chunksize=50000):
C:\Users\karle\AppData\Local\Temp\ipykernel_33184\1032639278.py:22: DtypeWarning: Columns (0: STATUS) ha

(1753245, 19)
2010-01 2026-03


In [28]:
df = pd.concat(chunks, ignore_index=True)
df.to_csv("../data/raw/04_unemployment.csv", index=False)
print(df.shape)
print(df["REF_DATE"].min(), df["REF_DATE"].max())

(1753245, 19)
2010-01 2026-03


## CMHC

Downloaded directly from: https://www.cmhc-schl.gc.ca/professionals/housing-markets-data-and-research/housing-data/data-tables/mortgage-and-debt/mortgage-delinquency-rate-canada-provinces-cmas